# Evo 2 (7B) Ghost Detection: Spin Test + Context Tax

**Hypothesis**: Evo 2's stability across context-window scaling may be a **"Ghost"**
-- a geometrically perfect manifold that is not grounded in biological reality.

## Two Tests

### Test 1: Self-Procrustes "Spin" Test
For each context-window checkpoint, compute `Procrustes(Clean, Perturbed)`.

- **Low residual** = the perturbed manifold overlays the clean manifold after rotation.
  The embedding is **grounded** (fixed orientation).
- **High residual** = the model creates a perfect geometric shape, but it
  **spins wildly** in the latent space. It has no "North Star."

### Test 2: Frozen Head "Context Tax" Test
Train a simple linear classifier on Evo 2 embeddings for a task that only
requires **local information** (1 kbp). Pad signals to match each checkpoint's
context window.

## Evo 2 (7B) Checkpoints

| Checkpoint | Context | Label |
|---|---|---|
| `evo2_7b_base` | 8K | Evo2-7B-8K |
| `evo2_7b_262k` | 262K | Evo2-7B-262K |
| `evo2_7b` | 1M | Evo2-7B-1M |

## Prerequisites

Run after `Evo2_Scaling_Experiment.ipynb` or `Evo2_RealDNA_Experiment.ipynb`.
Uses cached `.npy` embeddings for Test 1. Test 2 requires live Vortex inference.

---

In [ ]:
# Install Dependencies

print('Installing dependencies...')
!pip install -q shesha-geometry matplotlib seaborn pandas scipy scikit-learn
!pip install -q ninja
import sys, os

# Pin torch to 2.7.1+cu128 (Arc Institute recommended for evo2)
print('Pinning torch to 2.7.1+cu128...')
!pip install -q torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128

import torch
print(f'torch {torch.__version__} | CUDA {torch.version.cuda}')

# Build flash-attn from source (~10-15 min on A100)
print('Building flash-attn 2.8.0.post2 from source (~10-15 min)...')
!pip install flash-attn==2.8.0.post2 --no-build-isolation

!pip install -q evo2

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from scipy.linalg import orthogonal_procrustes
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import seaborn as sns

matplotlib.rcParams.update({'font.size': 11})

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from evo2 import Evo2
print('\nEvo2 package loaded successfully')
print('Done!')

In [ ]:
# Configuration

# Paths to cached Evo 2 embeddings (from Scaling or RealDNA experiments)
SCALING_CACHE = './results/evo2_scaling_experiment/cache'
REALDNA_CACHE = './results/evo2_realdna_experiment/cache'

OUTPUT_BASE = './results/evo2_ghost_detection/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Evo 2 7B context-window checkpoints (scaling variable = context window)
CONTEXT_CHECKPOINTS = [
    ('evo2_7b_base', 7.0,   8192,     'Evo2-7B-8K'),
    ('evo2_7b_262k', 7.0,  262144,   'Evo2-7B-262K'),
    ('evo2_7b',      7.0,  1048576,  'Evo2-7B-1M'),
]

# Perturbations to include in Procrustes test
PERTURBATIONS = ['reverse_complement', 'snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct']

# Embedding layer for 7B models (Vortex naming)
EMBEDDING_LAYER = 'blocks.28.mlp.l3'

BATCH_OVERRIDES = {
    'evo2_7b_base': 4,
    'evo2_7b_262k': 4,
    'evo2_7b':      2,
}

# Frozen Head config
N_CLASSIFICATION_SEQS = 200  # per class (200 human + 200 yeast = 400 total)
SIGNAL_LENGTH = 1000  # 1 kbp signal region
N_CV_FOLDS = 5

print(f'Output: {RESULTS_DIR}')
print(f'Scaling cache: {SCALING_CACHE}')
print(f'RealDNA cache: {REALDNA_CACHE}')
print(f'Context checkpoints: {[lbl for _, _, _, lbl in CONTEXT_CHECKPOINTS]}')

In [ ]:
# Load Cached Embeddings for Test 1
#
# Scan both Scaling and RealDNA experiment caches.
# Prefer RealDNA (real genomic DNA) over Scaling (synthetic) if both exist.

def find_embedding(ckpt_name, suffix, source='realdna'):
    """Find a cached .npy embedding file for a given checkpoint."""
    if source == 'realdna':
        safe_name = ckpt_name.replace('/', '_').replace('-', '_') + '_realdna'
        base = REALDNA_CACHE
    else:
        safe_name = ckpt_name.replace('/', '_').replace('-', '_')
        base = SCALING_CACHE

    patterns = [
        f'{base}/{safe_name}_{suffix}.npy',
    ]
    for path in patterns:
        if os.path.exists(path):
            return path
    return None


# Discover what's available
print('Scanning cached embeddings...\n')
available_data = {}  # ckpt_name -> {'clean': path, 'reverse_complement': path, ...}

for ckpt_name, params_b, ctx_len, label in CONTEXT_CHECKPOINTS:
    data = {}

    # Try RealDNA first, then Scaling
    for source in ['realdna', 'scaling']:
        clean_path = find_embedding(ckpt_name, 'clean', source)
        if clean_path is not None:
            data['clean'] = clean_path
            data['source'] = source

            for pert in PERTURBATIONS:
                pert_path = find_embedding(ckpt_name, pert, source)
                if pert_path is not None:
                    data[pert] = pert_path
            break

    if 'clean' in data:
        available_data[label] = data
        n_perts = len([k for k in data if k not in ('clean', 'source')])
        X = np.load(data['clean'])
        print(f'  {label}: {X.shape[0]} samples x {X.shape[1]}d '
              f'({n_perts} perturbations, source={data["source"]})')
    else:
        print(f'  {label}: NOT FOUND (run Evo2 Scaling or RealDNA experiment first)')

print(f'\nFound data for: {list(available_data.keys())}')
if len(available_data) == 0:
    raise RuntimeError('No cached embeddings found! Run Evo2 experiments first.')

In [ ]:
# TEST 1 -- The Self-Procrustes "Spin" Test
#
# For each context-window checkpoint:
#   Raw Error   = ||X_clean - X_pert||_F
#   Aligned Error = Procrustes distance (best rotation + translation + scaling)
#   Ratio = Aligned / Raw
#   Residual = Aligned Error (normalized)
#
# KEY: If Residual(1M) > Residual(8K), the Ghost is detaching from reality.

def compute_procrustes_metrics(X_clean, X_pert, n_samples=5000):
    """Compute Procrustes ratio and residual between clean and perturbed."""
    if X_clean.shape[0] > n_samples:
        rng = np.random.default_rng(320)
        idx = rng.choice(X_clean.shape[0], n_samples, replace=False)
        X_clean = X_clean[idx]
        X_pert = X_pert[idx]

    X_c = X_clean - X_clean.mean(axis=0)
    X_p = X_pert - X_pert.mean(axis=0)

    raw_error = np.linalg.norm(X_c - X_p, 'fro') / np.sqrt(len(X_c))

    scale_c = np.linalg.norm(X_c, 'fro')
    scale_p = np.linalg.norm(X_p, 'fro')
    X_cn = X_c / scale_c
    X_pn = X_p / scale_p

    R, _ = orthogonal_procrustes(X_pn, X_cn)
    X_aligned = X_p @ R

    s = np.trace(X_c.T @ X_aligned) / np.trace(X_aligned.T @ X_aligned)
    X_aligned_scaled = X_aligned * s

    aligned_error = np.linalg.norm(X_c - X_aligned_scaled, 'fro') / np.sqrt(len(X_c))
    ratio = aligned_error / raw_error if raw_error > 1e-12 else 1.0
    reduction = 1.0 - ratio

    residual = np.linalg.norm(X_cn - X_pn @ R, 'fro') ** 2

    return {
        'raw_error': raw_error,
        'aligned_error': aligned_error,
        'ratio': ratio,
        'reduction_pct': reduction * 100,
        'optimal_scale': s,
        'residual': residual,
    }


# Run Procrustes on all available checkpoints x perturbations
print('=' * 70)
print('TEST 1: SELF-PROCRUSTES SPIN TEST (Evo 2 7B)')
print('=' * 70)

procrustes_rows = []

for ckpt_name, params_b, ctx_len, label in CONTEXT_CHECKPOINTS:
    if label not in available_data:
        print(f'\n--- {label} --- SKIPPED (no cached data)')
        continue

    data = available_data[label]
    X_clean = np.load(data['clean'])

    print(f'\n--- {label} ({ctx_len:,} ctx, n={X_clean.shape[0]}) ---')

    for pert in PERTURBATIONS:
        if pert not in data:
            continue
        X_pert = np.load(data[pert])

        result = compute_procrustes_metrics(X_clean, X_pert)
        procrustes_rows.append({
            'checkpoint': ckpt_name,
            'context_length': ctx_len,
            'context_label': label,
            'perturbation': pert,
            'raw_error': result['raw_error'],
            'aligned_error': result['aligned_error'],
            'ratio': result['ratio'],
            'reduction_pct': result['reduction_pct'],
            'optimal_scale': result['optimal_scale'],
            'residual': result['residual'],
        })

        print(f'  {pert:>25}: raw={result["raw_error"]:.4f}  '
              f'aligned={result["aligned_error"]:.4f}  '
              f'ratio={result["ratio"]:.3f}  '
              f'residual={result["residual"]:.4f}  '
              f'scale={result["optimal_scale"]:.4f}')

df_proc = pd.DataFrame(procrustes_rows)
df_proc.to_csv(f'{RESULTS_DIR}/ghost_procrustes.csv', index=False)
print(f'\nSaved {len(df_proc)} rows to {RESULTS_DIR}/ghost_procrustes.csv')

In [ ]:
# TEST 1 -- Visualization

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

PERT_COLORS = {
    'reverse_complement': '#DC2626',
    'snp_1pct': '#60A5FA',
    'snp_2pct': '#3B82F6',
    'snp_5pct': '#2563EB',
    'snp_10pct': '#1D4ED8',
}
CTX_ORDER = ['Evo2-7B-8K', 'Evo2-7B-262K', 'Evo2-7B-1M']

# Panel A: Procrustes Residual vs Context Length
ax = axes[0]
for pert in PERTURBATIONS:
    sub = df_proc[df_proc['perturbation'] == pert].copy()
    if sub.empty:
        continue
    sub['ctx_idx'] = sub['context_label'].map({c: i for i, c in enumerate(CTX_ORDER)})
    sub = sub.sort_values('ctx_idx')
    ax.plot(sub['ctx_idx'], sub['residual'],
            marker='o', linewidth=2, markersize=8,
            color=PERT_COLORS.get(pert, '#6B7280'),
            label=pert.replace('_', ' '))

ax.set_xticks(range(len(CTX_ORDER)))
ax.set_xticklabels(CTX_ORDER, fontsize=9)
ax.set_xlabel('Evo 2 Checkpoint (Context Window)')
ax.set_ylabel('Procrustes Residual (normalized)')
ax.set_title('A. "Spin" Test: Does the Ghost Detach?', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

# Panel B: Procrustes Ratio vs Context Length
ax = axes[1]
for pert in PERTURBATIONS:
    sub = df_proc[df_proc['perturbation'] == pert].copy()
    if sub.empty:
        continue
    sub['ctx_idx'] = sub['context_label'].map({c: i for i, c in enumerate(CTX_ORDER)})
    sub = sub.sort_values('ctx_idx')
    ax.plot(sub['ctx_idx'], sub['ratio'],
            marker='s', linewidth=2, markersize=8,
            color=PERT_COLORS.get(pert, '#6B7280'),
            label=pert.replace('_', ' '))

ax.set_xticks(range(len(CTX_ORDER)))
ax.set_xticklabels(CTX_ORDER, fontsize=9)
ax.set_xlabel('Evo 2 Checkpoint')
ax.set_ylabel('Procrustes Ratio (aligned / raw)')
ax.set_title('B. Ratio: How Much Can Rotation Fix?', fontweight='bold')
ax.axhline(y=1.0, color='#94A3B8', linestyle=':', linewidth=1, alpha=0.5)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

# Panel C: RC Residual vs All SNP Residuals (grouped bar)
ax = axes[2]
ctx_labels_present = [c for c in CTX_ORDER if c in available_data]
n_ctx = len(ctx_labels_present)
bar_width = 0.15
perts_present = [p for p in PERTURBATIONS if p in df_proc['perturbation'].values]

for j, pert in enumerate(perts_present):
    vals = []
    for ctx_label in ctx_labels_present:
        row = df_proc[(df_proc['context_label'] == ctx_label) & (df_proc['perturbation'] == pert)]
        vals.append(row['residual'].values[0] if len(row) > 0 else 0)
    x = np.arange(n_ctx) + j * bar_width
    ax.bar(x, vals, width=bar_width, color=PERT_COLORS.get(pert, '#6B7280'),
           label=pert.replace('_', ' '), edgecolor='white', linewidth=0.8)

ax.set_xticks(np.arange(n_ctx) + bar_width * (len(perts_present) - 1) / 2)
ax.set_xticklabels(ctx_labels_present, fontsize=9)
ax.set_xlabel('Evo 2 Checkpoint')
ax.set_ylabel('Procrustes Residual')
ax.set_title('C. Residual by Perturbation Type', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.15, axis='y')

fig.suptitle('Evo 2 (7B) Ghost Detection: Self-Procrustes Spin Test',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/ghost_spin_test.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved {RESULTS_DIR}/ghost_spin_test.png')

# Summary table
print('\n' + '=' * 70)
print('RC SPIN TEST SUMMARY (Reverse Complement only)')
print('=' * 70)
rc_df = df_proc[df_proc['perturbation'] == 'reverse_complement'].copy()
if not rc_df.empty:
    for _, row in rc_df.iterrows():
        print(f'  {row["context_label"]:>15}: residual={row["residual"]:.4f}  '
              f'ratio={row["ratio"]:.3f}  reduction={row["reduction_pct"]:.1f}%')
    if len(rc_df) >= 2:
        first = rc_df.iloc[0]
        last = rc_df.iloc[-1]
        if last['residual'] > first['residual'] * 1.1:
            print(f'\n  VERDICT: Ghost DETACHING -- residual grows '
                  f'{last["residual"]/first["residual"]:.2f}x from '
                  f'{first["context_label"]} to {last["context_label"]}')
        elif last['residual'] < first['residual'] * 0.9:
            print(f'\n  VERDICT: Ghost GROUNDING -- residual shrinks '
                  f'{first["residual"]/last["residual"]:.2f}x')
        else:
            print(f'\n  VERDICT: Ghost NEUTRAL -- residual roughly constant')
else:
    print('  No RC data found.')

In [ ]:
# TEST 2 Setup -- Evo 2 Vortex Embedding Function
#
# Only needed if running Test 2 (Frozen Head).
# If you only want Test 1 (Procrustes), you can skip this cell and below.

import gc
import time
import urllib.request
import gzip


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def make_evo2_embed_fn(checkpoint_name):
    """Load Evo 2 7B locally via Vortex. Extract hidden states."""
    print(f'Loading {checkpoint_name} via Vortex...')
    device = 'cuda:0'
    evo2_model = Evo2(checkpoint_name)
    n_params = sum(p.numel() for p in evo2_model.model.parameters()) / 1e6
    print(f'  Params: {n_params:.1f}M')
    print(f'  Embedding layer: {EMBEDDING_LAYER}')

    # Vortex FIR engine uses 32-bit index math.
    # Max safe length depends on hidden_dim; for 7B (d=4096),
    # safe limit is ~131K tokens. We cap at 120K to be safe.
    MAX_SAFE_SEQ_LEN = 120_000

    @torch.no_grad()
    def embed(sequences, signal_length=None):
        embeddings = []
        n_total = len(sequences)
        for idx, seq in enumerate(sequences):
            if (idx + 1) % 10 == 0 or idx == 0 or idx == n_total - 1:
                print(f'    Seq {idx+1}/{n_total}', end='\r', flush=True)

            # Truncate to avoid 32-bit overflow in Vortex FIR
            if len(seq) > MAX_SAFE_SEQ_LEN:
                seq_truncated = seq[:MAX_SAFE_SEQ_LEN]
            else:
                seq_truncated = seq

            input_ids = torch.tensor(
                evo2_model.tokenizer.tokenize(seq_truncated), dtype=torch.int,
            ).unsqueeze(0).to(device)
            _, layer_embs = evo2_model(
                input_ids, return_embeddings=True, layer_names=[EMBEDDING_LAYER],
            )
            emb = layer_embs[EMBEDDING_LAYER]  # (1, seq_len, hidden_dim)
            if signal_length is not None:
                # LOCAL POOLING: only pool signal region
                n_positions = emb.shape[1]
                scale = n_positions / len(seq_truncated)
                crop_end = max(1, int(signal_length * scale))
                pooled = emb[:, :crop_end, :].mean(dim=1).squeeze(0)
            else:
                pooled = emb.mean(dim=1).squeeze(0)
            embeddings.append(pooled.cpu().float().numpy())
        print()
        return np.array(embeddings, dtype=np.float32)

    return embed, evo2_model, n_params


print('Evo 2 embedding function ready')

In [ ]:
# TEST 2 -- Generate Classification Sequences
#
# Task: Distinguish real human DNA (chr22) from real yeast DNA.
# Both are real biological sequences, but from different species.
# A 1kbp region contains enough species-specific motifs for classification.
#
# Padding strategy: place the 1kbp signal at the START, pad the rest with random DNA.

VALID_BASES = set('ACGT')
DNA_BASES = ['A', 'C', 'G', 'T']

CHR22_URL = 'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr22.fa.gz'
YEAST_URL = 'https://hgdownload.soe.ucsc.edu/goldenPath/sacCer3/bigZips/sacCer3.fa.gz'


def download_genome_sequences(url, n_sequences, seq_length, seed, cache_tag):
    """Download a genome FASTA and sample non-overlapping regions."""
    cache_file = f'{CACHE_DIR}/{cache_tag}_{n_sequences}_{seq_length}.txt'
    if os.path.exists(cache_file):
        print(f'  Loading cached: {cache_file}')
        with open(cache_file) as f:
            return [line.strip() for line in f if line.strip()]

    print(f'  Downloading genome...')
    with urllib.request.urlopen(url) as response:
        raw = gzip.decompress(response.read()).decode('ascii')

    genome = []
    for line in raw.split('\n'):
        if line.startswith('>') or not line.strip():
            continue
        genome.append(line.strip().upper())
    full_seq = ''.join(genome)
    print(f'  Genome length: {len(full_seq):,} bp')

    rng = np.random.default_rng(seed)
    sequences = []
    max_start = len(full_seq) - seq_length
    attempts = 0
    while len(sequences) < n_sequences and attempts < n_sequences * 20:
        start = rng.integers(0, max_start)
        region = full_seq[start:start + seq_length]
        if all(b in VALID_BASES for b in region):
            sequences.append(region)
        attempts += 1

    os.makedirs(CACHE_DIR, exist_ok=True)
    with open(cache_file, 'w') as f:
        f.write('\n'.join(sequences))
    print(f'  Sampled {len(sequences)} sequences, cached to {cache_file}')
    return sequences


def pad_sequence(signal_seq, target_length, rng):
    """Pad a signal sequence with random DNA to reach target_length."""
    pad_needed = target_length - len(signal_seq)
    if pad_needed <= 0:
        return signal_seq[:target_length]
    padding = ''.join(rng.choice(DNA_BASES, size=pad_needed))
    return signal_seq + padding


print('Downloading classification sequences...')
print('\n--- Human (chr22) ---')
human_seqs = download_genome_sequences(
    CHR22_URL, N_CLASSIFICATION_SEQS, SIGNAL_LENGTH, seed=320, cache_tag='human_chr22'
)
print(f'\n--- Yeast (sacCer3) ---')
yeast_seqs = download_genome_sequences(
    YEAST_URL, N_CLASSIFICATION_SEQS, SIGNAL_LENGTH, seed=1991, cache_tag='yeast_saccer3'
)

print(f'\nHuman: {len(human_seqs)} sequences x {len(human_seqs[0])} bp')
print(f'Yeast: {len(yeast_seqs)} sequences x {len(yeast_seqs[0])} bp')

all_signals = human_seqs + yeast_seqs
labels = np.array([0] * len(human_seqs) + [1] * len(yeast_seqs))
print(f'Total: {len(all_signals)} sequences, classes: {np.bincount(labels)}')

In [ ]:
# TEST 2 -- Embed Padded Sequences at Each Context Window
#
# For each 7B checkpoint (different context windows):
#   1. Pad the 1kbp signal to the checkpoint's max context length
#   2. Embed via Evo 2 Vortex (LOCAL POOLING: only pool signal region)
#   3. Cache the embeddings

rng_pad = np.random.default_rng(320)
frozen_head_embeddings = {}  # label -> np.array

for ckpt_name, params_b, ctx_len, label in CONTEXT_CHECKPOINTS:
    cache_path = f'{CACHE_DIR}/frozen_head_{label}.npy'

    if os.path.exists(cache_path):
        print(f'\n{label}: Loading cached embeddings')
        frozen_head_embeddings[label] = np.load(cache_path)
        continue

    print(f'\n{"=" * 70}')
    print(f'{label}: Padding {len(all_signals)} signals to {ctx_len:,} bp')
    print('=' * 70)

    # Pad signals to target context length
    # padded = [pad_sequence(s, ctx_len, rng_pad) for s in all_signals]
    # print(f'  Padded sequence length: {len(padded[0]):,} bp')
    # print(f'  Signal fraction: {SIGNAL_LENGTH/ctx_len*100:.2f}%')

    # Pad signals to target context length, then truncate to safe CUDA limit
    MAX_SAFE_SEQ_LEN = 120_000
    target_len = min(ctx_len, MAX_SAFE_SEQ_LEN)
    padded = [pad_sequence(s, target_len, rng_pad) for s in all_signals]
    print(f'  Padded sequence length: {len(padded[0]):,} bp')
    print(f'  Original context window: {ctx_len:,} bp (truncated to {target_len:,})')
    print(f'  Signal fraction: {SIGNAL_LENGTH/target_len*100:.2f}%')

    # Load model and embed
    embed_fn, evo2_model, n_params = make_evo2_embed_fn(ckpt_name)
    print(f'  Embedding via Evo 2 Vortex (LOCAL POOL: first {SIGNAL_LENGTH} positions)...')
    embs = embed_fn(padded, signal_length=SIGNAL_LENGTH)
    np.save(cache_path, embs)
    print(f'  Cached to {cache_path}')
    frozen_head_embeddings[label] = embs

    # Global pooling (for comparison)
    cache_global = f'{CACHE_DIR}/frozen_head_global_{label}.npy'
    if not os.path.exists(cache_global):
        print(f'  Embedding (GLOBAL POOL: all {ctx_len:,} positions)...')
        embs_g = embed_fn(padded, signal_length=None)
        np.save(cache_global, embs_g)

    # Free GPU
    del evo2_model, embed_fn
    cleanup_gpu()

print(f'\nEmbeddings ready for {list(frozen_head_embeddings.keys())}')

In [ ]:
# TEST 2 -- Train Frozen Linear Classifiers
#
# For each context checkpoint:
#   Train LogisticRegression on Evo 2 embeddings
#   Report cross-validated accuracy

print('=' * 70)
print('TEST 2: FROZEN HEAD -- Context Tax Analysis (Evo 2 7B)')
print('=' * 70)

frozen_head_rows = []

for ckpt_name, params_b, ctx_len, label in CONTEXT_CHECKPOINTS:
    if label not in frozen_head_embeddings:
        print(f'\n{label}: SKIPPED (no embeddings)')
        continue

    X = frozen_head_embeddings[label]
    y = labels[:X.shape[0]]

    print(f'\n--- {label} ({ctx_len:,} ctx) ---')
    print(f'  Embedding shape: {X.shape}')
    print(f'  Signal fraction: {SIGNAL_LENGTH/ctx_len*100:.3f}%')

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, C=1.0, random_state=320)
    )

    cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=320)
    scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')

    print(f'  Accuracy: {scores.mean():.4f} \u00b1 {scores.std():.4f}')
    print(f'  Per-fold: {[f"{s:.3f}" for s in scores]}')

    frozen_head_rows.append({
        'checkpoint': ckpt_name,
        'context_length': ctx_len,
        'context_label': label,
        'signal_fraction': SIGNAL_LENGTH / ctx_len,
        'accuracy_mean': scores.mean(),
        'accuracy_std': scores.std(),
        'accuracy_min': scores.min(),
        'accuracy_max': scores.max(),
    })

df_frozen = pd.DataFrame(frozen_head_rows)
df_frozen.to_csv(f'{RESULTS_DIR}/ghost_frozen_head.csv', index=False)

# Verdict
print(f'\n{"=" * 70}')
print('FROZEN HEAD VERDICT')
print('=' * 70)
if len(df_frozen) >= 2:
    first = df_frozen.iloc[0]
    for _, row in df_frozen.iloc[1:].iterrows():
        delta = row['accuracy_mean'] - first['accuracy_mean']
        ctx_ratio = row['context_length'] / first['context_length']
        print(f'  {first["context_label"]} -> {row["context_label"]} '
              f'({ctx_ratio:.0f}x context): '
              f'\u0394accuracy = {delta:+.4f}')
        if delta < -0.05:
            print(f'    -> TOXIC: Extra context HURTS ({delta:.1%} drop). Ghost confirmed.')
        elif abs(delta) < 0.02:
            print(f'    -> DIMINISHING RETURNS: {ctx_ratio:.0f}x more data for \u22480 gain.')
        else:
            print(f'    -> HELPFUL: Extra context aids classification.')

print(f'\nSaved to {RESULTS_DIR}/ghost_frozen_head.csv')

In [ ]:
# TEST 2 -- Visualization

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ctx_labels_fh = df_frozen['context_label'].tolist()
ctx_indices = np.arange(len(ctx_labels_fh))

# Panel A: Accuracy vs Context Length
ax = axes[0]
ax.errorbar(ctx_indices, df_frozen['accuracy_mean'], yerr=df_frozen['accuracy_std'],
            marker='o', linewidth=2.5, markersize=10, capsize=6,
            color='#7C3AED', ecolor='#C4B5FD', elinewidth=2)
ax.axhline(y=0.5, color='#DC2626', linestyle='--', linewidth=1, alpha=0.7, label='Random chance')

for i, row in df_frozen.iterrows():
    ax.annotate(f'{row["signal_fraction"]*100:.1f}% signal',
                xy=(i, row['accuracy_mean']),
                xytext=(0, -20), textcoords='offset points',
                fontsize=8, color='#6B7280', ha='center')

ax.set_xticks(ctx_indices)
ax.set_xticklabels(ctx_labels_fh, fontsize=9)
ax.set_xlabel('Evo 2 Checkpoint (Context Window)', fontsize=11)
ax.set_ylabel('Frozen Head Accuracy (Human vs Yeast)', fontsize=11)
ax.set_title('A. Context Tax: Does More Context Help?', fontweight='bold', fontsize=12)
ax.set_ylim(0.4, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)

# Panel B: Signal Fraction vs Accuracy
ax = axes[1]
ax.scatter(df_frozen['signal_fraction'] * 100, df_frozen['accuracy_mean'],
           s=200, c='#7C3AED', zorder=5, edgecolors='white', linewidths=2)
for i, row in df_frozen.iterrows():
    ax.annotate(row['context_label'],
                xy=(row['signal_fraction'] * 100, row['accuracy_mean']),
                xytext=(10, 5), textcoords='offset points',
                fontsize=10, fontweight='bold')

ax.set_xscale('log')
ax.set_xlabel('Signal Fraction (% of input that is real)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('B. Signal Dilution Curve', fontweight='bold', fontsize=12)
ax.axhline(y=0.5, color='#DC2626', linestyle='--', linewidth=1, alpha=0.7)
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.15)

fig.suptitle('Evo 2 (7B) Ghost Detection: Frozen Head Context Tax',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/ghost_frozen_head.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved {RESULTS_DIR}/ghost_frozen_head.png')

In [ ]:
# Combined Figure -- Spin Test + Context Tax

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel A: RC Procrustes Residual vs Context
ax = axes[0]
rc_df = df_proc[df_proc['perturbation'] == 'reverse_complement'].copy()
if not rc_df.empty:
    rc_df['ctx_idx'] = rc_df['context_label'].map({c: i for i, c in enumerate(CTX_ORDER)})
    rc_df = rc_df.sort_values('ctx_idx')
    ax.plot(rc_df['ctx_idx'], rc_df['residual'],
            marker='D', linewidth=2.5, markersize=10, color='#DC2626')
    ax.fill_between(rc_df['ctx_idx'], 0, rc_df['residual'], color='#DC2626', alpha=0.1)

ax.set_xticks(range(len(CTX_ORDER)))
ax.set_xticklabels(CTX_ORDER, fontsize=9)
ax.set_xlabel('Evo 2 Checkpoint')
ax.set_ylabel('Procrustes Residual (RC vs Clean)')
ax.set_title('A. Spin Test: RC Orientation Drift', fontweight='bold')
ax.grid(True, alpha=0.15)

# Panel B: Frozen Head Accuracy vs Context
ax = axes[1]
if len(df_frozen) > 0:
    ctx_fh = np.arange(len(df_frozen))
    ax.errorbar(ctx_fh, df_frozen['accuracy_mean'], yerr=df_frozen['accuracy_std'],
                marker='o', linewidth=2.5, markersize=10, capsize=6,
                color='#7C3AED', ecolor='#C4B5FD', elinewidth=2)
    ax.set_xticks(ctx_fh)
    ax.set_xticklabels(df_frozen['context_label'].tolist(), fontsize=9)
    ax.axhline(y=0.5, color='#DC2626', linestyle='--', linewidth=1, alpha=0.5)

ax.set_xlabel('Evo 2 Checkpoint')
ax.set_ylabel('Frozen Head Accuracy')
ax.set_title('B. Context Tax: Classification Utility', fontweight='bold')
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.15)

# Panel C: Dual-axis -- Residual + Accuracy on same plot
ax1 = axes[2]
ax2 = ax1.twinx()

if not rc_df.empty:
    ax1.plot(rc_df['ctx_idx'], rc_df['residual'],
             marker='D', linewidth=2.5, markersize=9, color='#DC2626',
             label='Procrustes Residual (RC)')
    ax1.set_ylabel('Procrustes Residual', color='#DC2626')
    ax1.tick_params(axis='y', labelcolor='#DC2626')

if len(df_frozen) > 0:
    fh_ctx_idx = [CTX_ORDER.index(c) for c in df_frozen['context_label'] if c in CTX_ORDER]
    ax2.plot(fh_ctx_idx, df_frozen['accuracy_mean'].values[:len(fh_ctx_idx)],
             marker='o', linewidth=2.5, markersize=9, color='#7C3AED',
             label='Frozen Head Accuracy')
    ax2.set_ylabel('Accuracy', color='#7C3AED')
    ax2.tick_params(axis='y', labelcolor='#7C3AED')
    ax2.set_ylim(0.4, 1.05)
    ax2.axhline(y=0.5, color='#7C3AED', linestyle=':', linewidth=1, alpha=0.3)

ax1.set_xticks(range(len(CTX_ORDER)))
ax1.set_xticklabels(CTX_ORDER, fontsize=9)
ax1.set_xlabel('Evo 2 Checkpoint')
ax1.set_title('C. Combined: Spin vs. Utility', fontweight='bold')
ax1.grid(True, alpha=0.15)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='lower left')

fig.suptitle('Evo 2 (7B) Ghost Detection: Combined Evidence',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/ghost_combined.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved {RESULTS_DIR}/ghost_combined.png')